# Week 1 — TikTok fixed-dollar strategy

This notebook tests the strategy on **every stock available in `data/egx`**:

- If a stock fell **5% or more** over the previous five trading sessions, buy **$5** of it.
- If a stock rose **10% or more** over the previous five trading sessions, sell **$10** of it.
- Otherwise, do nothing.

The signal is calculated using yesterday's close and executed at today's close. This one-session delay prevents look-ahead bias. The simulation allows fractional shares and never shorts a stock: a sell is capped at the value currently held. Fixed-dollar orders require a cash-and-shares ledger, so this strategy is intentionally implemented directly rather than converted into portfolio weights.

> Note: the source data is for EGX securities. The `$5` and `$10` instructions are treated as account-currency units without an FX conversion.

In [ ]:
import os
import sys

while not os.path.isdir('src') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
sys.path.insert(0, 'src')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tradinglab.data_feed import DataFeed

feed = DataFeed.from_dir('data/egx')
print(f'Loaded {feed.n_assets} stocks and {feed.n_days} shared trading days.')
print(feed.symbols)

## Parameters

`COMMISSION = 0.005` charges 0.5% of each buy or sell, matching the dashboard assumption. Change it to `0.0` to inspect a frictionless result.

In [ ]:
LOOKBACK = 5
BUY_THRESHOLD = -0.05
SELL_THRESHOLD = 0.10
BUY_AMOUNT = 5.0
SELL_AMOUNT = 10.0
INITIAL_CASH = 1_000.0
COMMISSION = 0.005

print(f'Buy ${BUY_AMOUNT:.0f} after a weekly return <= {BUY_THRESHOLD:.0%}')
print(f'Sell ${SELL_AMOUNT:.0f} after a weekly return >= {SELL_THRESHOLD:.0%}')
print(f'Commission: {COMMISSION:.1%} per transaction')

## Build the weekly signals

A five-session return compares each close with its close five trading sessions earlier. The signal matrix is shifted by one row, so a move known at the end of day `t-1` can only trade on day `t`.

In [ ]:
weekly_returns = np.full_like(feed.close, np.nan, dtype=float)
weekly_returns[LOOKBACK:] = feed.close[LOOKBACK:] / feed.close[:-LOOKBACK] - 1.0

buy_signal = np.zeros_like(feed.close, dtype=bool)
sell_signal = np.zeros_like(feed.close, dtype=bool)
buy_signal[1:] = weekly_returns[:-1] <= BUY_THRESHOLD
sell_signal[1:] = weekly_returns[:-1] >= SELL_THRESHOLD

print('Buy signals :', int(buy_signal.sum()))
print('Sell signals:', int(sell_signal.sum()))

## Backtest with cash and fractional shares

Sells run before buys each day so their proceeds are available. When there is not enough cash for every $5 order, the remaining cash is divided equally among that day's buy signals. Commission is deducted immediately.

In [ ]:
def run_fixed_dollar_strategy(
    feed,
    buy_signal,
    sell_signal,
    initial_cash=INITIAL_CASH,
    buy_amount=BUY_AMOUNT,
    sell_amount=SELL_AMOUNT,
    commission=COMMISSION,
):
    cash = float(initial_cash)
    shares = np.zeros(feed.n_assets, dtype=float)
    equity = np.empty(feed.n_days, dtype=float)
    cash_history = np.empty(feed.n_days, dtype=float)
    trades = []

    for day in range(feed.n_days):
        prices = feed.close[day]

        # Sell first. The gross order is capped at the position's value.
        for asset in np.flatnonzero(sell_signal[day]):
            gross = min(sell_amount, shares[asset] * prices[asset])
            if gross <= 0:
                continue
            fee = gross * commission
            shares[asset] -= gross / prices[asset]
            cash += gross - fee
            trades.append({
                'date': feed.dates[day], 'symbol': feed.symbols[asset],
                'side': 'SELL', 'gross': gross, 'fee': fee,
                'price': prices[asset],
            })

        # Give every buy signal the same order size if cash is constrained.
        buy_assets = np.flatnonzero(buy_signal[day])
        if len(buy_assets):
            gross_each = min(buy_amount, cash / (len(buy_assets) * (1.0 + commission)))
            for asset in buy_assets:
                if gross_each <= 0:
                    break
                fee = gross_each * commission
                shares[asset] += gross_each / prices[asset]
                cash -= gross_each + fee
                trades.append({
                    'date': feed.dates[day], 'symbol': feed.symbols[asset],
                    'side': 'BUY', 'gross': gross_each, 'fee': fee,
                    'price': prices[asset],
                })

        cash = max(cash, 0.0)  # remove tiny floating-point negatives
        cash_history[day] = cash
        equity[day] = cash + np.dot(shares, prices)

    trade_log = pd.DataFrame(trades)
    return {
        'equity': equity,
        'cash': cash_history,
        'shares': shares,
        'trades': trade_log,
    }

result = run_fixed_dollar_strategy(feed, buy_signal, sell_signal)
trade_log = result['trades']
print(f"Trades: {len(trade_log):,}")
print(f"Final equity: ${result['equity'][-1]:,.2f}")
print(f"Final cash:   ${result['cash'][-1]:,.2f}")
print(f"Commission paid: ${trade_log['fee'].sum():,.2f}")

## Result versus the full-market benchmark

In [ ]:
benchmark_returns = feed.returns.mean(axis=1)
benchmark = INITIAL_CASH * np.cumprod(1.0 + benchmark_returns)

strategy_return = result['equity'][-1] / INITIAL_CASH - 1.0
benchmark_return = benchmark[-1] / INITIAL_CASH - 1.0
running_peak = np.maximum.accumulate(result['equity'])
max_drawdown = np.max(1.0 - result['equity'] / running_peak)

summary = pd.Series({
    'stocks': feed.n_assets,
    'buy trades': int((trade_log['side'] == 'BUY').sum()),
    'sell trades': int((trade_log['side'] == 'SELL').sum()),
    'strategy return': strategy_return,
    'benchmark return': benchmark_return,
    'max drawdown': max_drawdown,
    'commission paid': trade_log['fee'].sum(),
})
display(summary.to_frame('value'))

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 9), sharex=True, height_ratios=[3, 1])
ax1.plot(feed.dates, result['equity'], label='TikTok fixed-dollar strategy', linewidth=1.5)
ax1.plot(feed.dates, benchmark, label='Equal-weight full market', linewidth=1.2, alpha=0.85)
ax1.set_title('Strategy equity vs full-market benchmark')
ax1.set_ylabel('Account value ($)')
ax1.legend()
ax1.grid(alpha=0.25)

ax2.plot(feed.dates, result['cash'], color='tab:gray', linewidth=1.1)
ax2.set_title('Cash available')
ax2.set_ylabel('$')
ax2.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## Check that every market symbol was considered

The table reports signal and completed-trade counts for every symbol. A zero count means that stock never crossed the corresponding threshold during the shared history, not that it was excluded.

In [ ]:
activity = pd.DataFrame({
    'symbol': feed.symbols,
    'buy_signals': buy_signal.sum(axis=0),
    'sell_signals': sell_signal.sum(axis=0),
}).set_index('symbol')

completed = trade_log.groupby(['symbol', 'side']).size().unstack(fill_value=0)
activity = activity.join(completed.rename(columns={'BUY': 'buys_completed', 'SELL': 'sells_completed'}))
activity = activity.fillna(0).astype(int)
display(activity)

print('Most recent 20 trades:')
display(trade_log.tail(20))

## Interpretation

This is a mechanical threshold strategy, not a recommendation to trade. Its most important limitations are fractional-share execution, no bid/ask spread or market impact, account-currency ambiguity for EGX prices, and repeated daily orders while a weekly threshold remains true. Compare the result with the benchmark and the total fees before deciding whether the rule is useful.